# ResNet1D Optimization with Optuna (Persistent Storage)
This notebook uses Optuna's SQLite storage to save trial results to Google Drive, allowing you to resume optimization across multiple Colab sessions.


## 1. Setup & Install Dependencies

In [ ]:
!pip install optuna torch scikit-learn pandas numpy -q
print("Dependencies installed!")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

## 3. Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

# ===== CONFIG =====
CSV_PATH = "/content/drive/MyDrive/MQP/Data/cleaned_data_but_in_rows.csv"
SPECIES_COL = "Species"

# Optuna storage (persistent across runs)
STUDY_NAME = "resnet1d_optimization"
STORAGE_PATH = "/content/drive/MyDrive/MQP/optuna_studies"
os.makedirs(STORAGE_PATH, exist_ok=True)
STORAGE_URL = f"sqlite:///{STORAGE_PATH}/{STUDY_NAME}.db"

# Model optimization parameters
N_TRIALS = 200  # Total trials to run (can be resumed)
NUM_EPOCHS = 100
PATIENCE = 15
RANDOM_STATE = 8

# Output paths
MODEL_OUT = "/content/drive/MyDrive/MQP/best_resnet_model.pth"
RESULTS_OUT = "/content/drive/MyDrive/MQP/resnet_optimization_results.csv"

print(f"Study Name: {STUDY_NAME}")
print(f"Storage Path: {STORAGE_URL}")
print(f"CSV Data: {CSV_PATH}")

## 4. Load & Preprocess Data

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv(CSV_PATH)
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSpecies distribution:")
print(df[SPECIES_COL].value_counts())

# Prepare features and labels
X_raw = df.drop(columns=[SPECIES_COL])
y = df[SPECIES_COL].astype(str).values

# Normalize X
X = (X_raw - X_raw.mean().values) / (X_raw.std().values + 1e-8)
X = X.fillna(0).astype(np.float32).values
X = np.expand_dims(X, axis=1)  # Add channel dimension for 1D conv

# Encode species labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)

print(f"\nFeatures shape: {X.shape}")
print(f"Number of classes: {n_classes}")
print(f"Classes: {le.classes_}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 5. Define ResNet1D Model

In [ ]:
class ResidualBlock1D(nn.Module):
    """Residual block for 1d convolution."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, downsample=None):
        super(ResidualBlock1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                              stride=stride, padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size,
                              stride=1, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out


class ResNet1D(nn.Module):
    """1d residual network for sequence processing."""
    def __init__(self, num_classes, input_channels=1, initial_filters=64, dropout=0.5):
        super(ResNet1D, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, initial_filters, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(initial_filters)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(initial_filters, initial_filters, blocks=2)
        self.layer2 = self._make_layer(initial_filters, initial_filters*2, blocks=2, stride=2)
        self.layer3 = self._make_layer(initial_filters*2, initial_filters*4, blocks=2, stride=2)
        self.layer4 = self._make_layer(initial_filters*4, initial_filters*8, blocks=2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(initial_filters*8, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        layers = [ResidualBlock1D(in_channels, out_channels, stride=stride, downsample=downsample)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock1D(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

print("ResNet1D model defined!")

## 6. Training Function

In [ ]:
import torch.optim as optim

def train_resnet(X, y, initial_filters, dropout, learning_rate, batch_size, weight_decay):
    """Train ResNet1D with 5-fold cross-validation and early stopping."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []

    X_tensor = torch.FloatTensor(X).to(DEVICE)
    y_tensor = torch.LongTensor(y).to(DEVICE)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train = X_tensor[train_idx]
        y_train = y_tensor[train_idx]
        X_val = X_tensor[val_idx]
        y_val = y_tensor[val_idx]

        # Create dataloaders
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # Model
        model = ResNet1D(
            num_classes=n_classes,
            input_channels=1,
            initial_filters=initial_filters,
            dropout=dropout
        ).to(DEVICE)

        # Optimizer
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )
        criterion = nn.CrossEntropyLoss()

        # Early stopping
        best_val_acc = 0.0
        patience_counter = 0
        min_val_loss = float('inf')

        for epoch in range(NUM_EPOCHS):
            # Train
            model.train()
            train_loss = 0.0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # Validate
            model.eval()
            with torch.no_grad():
                outputs = model(X_val)
                val_preds = torch.argmax(outputs, dim=1)
                val_acc = (val_preds == y_val).float().mean().item()
                val_loss = criterion(outputs, y_val).item()

            # Update learning rate
            scheduler.step(val_loss)

            # Track best validation accuracy
            if val_acc > best_val_acc:
                best_val_acc = val_acc

            # Early stopping
            if val_loss < min_val_loss:
                min_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                break

        fold_scores.append(best_val_acc)

    return np.mean(fold_scores)

print("Training function defined!")

## 7. Baseline Model (Pre-Optuna)

In [ ]:
print("="*60)
print("BASELINE: ResNet1D (Pre-Optuna)")
print("="*60)
print("\nBaseline hyperparameters:")
print("  initial_filters: 64")
print("  dropout: 0.5")
print("  learning_rate: 0.001")
print("  batch_size: 32")
print("  weight_decay: 0.0001")

baseline_acc = train_resnet(
    X, y_encoded,
    initial_filters=64,
    dropout=0.5,
    learning_rate=0.001,
    batch_size=32,
    weight_decay=0.0001
)

print(f"\n{'='*60}")
print(f"Baseline CV Accuracy: {baseline_acc:.4f}")
print(f"{'='*60}")

## 8. Define Optuna Objective Function

In [ ]:
import optuna

def objective(trial):
    """
    Optuna objective function for ResNet1D hyperparameter optimization.
    Returns: cross-validation accuracy (higher is better)
    """
    # Hyperparameters to optimize
    initial_filters = trial.suggest_int('initial_filters', 32, 128, step=16)
    dropout = trial.suggest_float('dropout', 0.1, 0.7)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    cv_score = train_resnet(X, y_encoded, initial_filters, dropout, learning_rate, batch_size, weight_decay)
    return cv_score

print("Objective function defined!")

## 9. Create or Load Optuna Study

In [ ]:
from optuna.storages import RDBStorage

# Create or load study
storage = RDBStorage(STORAGE_URL)
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=storage,
    study_name=STUDY_NAME,
    load_if_exists=True
)

print(f"Study loaded: {STUDY_NAME}")
print(f"Completed trials: {len(study.trials)}")
if len(study.trials) > 0:
    print(f"Best value so far: {study.best_value:.4f}")
    print(f"\nBest hyperparameters:")
    for key, value in study.best_params.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.6f}")
        else:
            print(f"  {key}: {value}")

## 10. Execute Trials (Resumable)

In [ ]:
import time

# How many new trials to run this session
NEW_TRIALS = 10  # Adjust based on time available

print(f"Running {NEW_TRIALS} trials...\n")
start_time = time.time()

try:
    study.optimize(objective, n_trials=NEW_TRIALS, show_progress_bar=True)
except KeyboardInterrupt:
    print("\nOptimization interrupted by user.")
except Exception as e:
    print(f"\nError during optimization: {e}")

elapsed_time = time.time() - start_time
print(f"\nCompleted in {elapsed_time/60:.1f} minutes")
print(f"Total trials completed: {len(study.trials)}")
print(f"Best value: {study.best_value:.4f}")

## 11. View Results & Best Hyperparameters

In [ ]:
import pandas as pd

# Best trial
print("=" * 60)
print("BEST TRIAL")
print("=" * 60)
print(f"Trial Number: {study.best_trial.number}")
print(f"CV Accuracy: {study.best_value:.4f}")
print(f"\nBest Hyperparameters:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

# Export trial history to CSV
trials_df = study.trials_dataframe()
trials_df['value'] = trials_df['value'].apply(lambda x: x * 100 if x is not None else None)
trials_df.to_csv(RESULTS_OUT, index=False)
print(f"\nResults saved to: {RESULTS_OUT}")
print(f"Total trials in history: {len(trials_df)}")

# Show top 5 trials
print("\n" + "="*60)
print("TOP 5 TRIALS")
print("="*60)
top_trials = trials_df.nlargest(5, "value")[["number", "value", "state"]]
print(top_trials.to_string())

## 12. Train Best Model & Evaluate

In [ ]:
# Build best model with best hyperparameters
best_params = study.best_params
print("="*60)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*60)

# Train on all data
X_tensor = torch.FloatTensor(X).to(DEVICE)
y_tensor = torch.LongTensor(y_encoded).to(DEVICE)

train_dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)

# Model
final_model = ResNet1D(
    num_classes=n_classes,
    input_channels=1,
    initial_filters=best_params['initial_filters'],
    dropout=best_params['dropout']
).to(DEVICE)

# Optimizer
optimizer = optim.Adam(final_model.parameters(), 
                       lr=best_params['learning_rate'],
                       weight_decay=best_params['weight_decay'])
criterion = nn.CrossEntropyLoss()

# Training loop
print(f"\nTraining for {NUM_EPOCHS} epochs...")
for epoch in range(NUM_EPOCHS):
    final_model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = final_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_loss/len(train_loader):.4f}")

# Save model
torch.save(final_model.state_dict(), MODEL_OUT)
print(f"\nBest model saved to: {MODEL_OUT}")

# COMPARISON WITH BASELINE
print(f"\n{'='*60}")
print("BASELINE vs OPTIMIZED COMPARISON")
print(f"{'='*60}")
print(f"Baseline CV Accuracy:      {baseline_acc:.4f}")
print(f"Optimized CV Accuracy:     {study.best_value:.4f}")
improvement = (study.best_value - baseline_acc) * 100
print(f"\nImprovement over baseline: {improvement:+.2f}%")
print(f"{'='*60}")

## Notes for Resuming Optimization

**To resume optimization in a new Colab session:**

1. Run cells 1-6 (Setup, Mount Drive, Config, Load Data, Model, Training Function)
2. Run cell 7 (Baseline) - optional, can skip if already trained
3. Run cells 8-9 (Define Objective, Create Study)
   - The study will automatically load existing trials from the SQLite database
4. Run cell 10 (Execute Trials) to continue optimization
5. Run cells 11-12 to view results and train the final model

The SQLite database persists on your Drive, so you won't lose any trial results!

**Key Points:**
- Baseline is ResNet1D with default hyperparameters
- Optuna optimizes: initial_filters, dropout, learning_rate, batch_size, weight_decay
- Uses 5-fold stratified cross-validation for robust scoring
- Each trial runs ~25-40 seconds with GPU (varies by hyperparams)
- With GPU, ~180-240 trials per 15-hour Colab session